In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load variables from .env
load_dotenv()

# Get API key
API_KEY = os.getenv("OPENAI_API_KEY")

MODEL = "gpt-4o-mini"

# Create OpenAI client
client = OpenAI(api_key=API_KEY)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        # {
        #     "role": "system",
        #     "content": "President of USA is Donald Trump. If any question is asked related to that then answer from here."
        # },
        {
            "role": "user",
            "content": "Who is the President of USA?"
        }
    ]
)

print(response)
print(response.choices[0].message.content)

In [ ]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# 1. OPENAI SETUP
# ============================================================

# Load variables from .env
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

if not API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. Please add it to your .env file."
    )

MODEL = "gpt-4o-mini"

client = OpenAI(api_key=API_KEY)


# ============================================================
# 2. STUFFING - SMALL INFORMATION KEPT INSIDE THE PROMPT
# ============================================================

return_policy = """
RETURN POLICY (updated 2026):
- Returns accepted within 30 days of delivery.
- Opened electronics: 15% restocking fee applies.
- Refunds are issued to the original payment method within 5-7 business days.
"""


# ============================================================
# 3. RAG - PRODUCT CATALOG
# ============================================================

product_catalog = [
    {
        "id": "P1",
        "name": "Compact 2-Burner Induction Stove",
        "tags": ["small kitchen", "compact", "cooking"],
    },
    {
        "id": "P2",
        "name": "6-Seater Dining Table",
        "tags": ["large kitchen", "dining", "furniture"],
    },
    {
        "id": "P3",
        "name": "Slim Under-Counter Fridge",
        "tags": ["small kitchen", "compact", "storage"],
    },
    {
        "id": "P4",
        "name": "Industrial 8-Burner Range",
        "tags": ["commercial kitchen", "large", "cooking"],
    },
    {
        "id": "P5",
        "name": "Wall-Mounted Fold-Down Table",
        "tags": ["small kitchen", "compact", "furniture"],
    },
]


def retrieve_relevant_products(question, top_k=3):
    """
    Simple keyword-based retrieval.

    In a real RAG system, this could be replaced with
    embeddings + a vector database.
    """

    question_lower = question.lower()
    scored_products = []

    for product in product_catalog:
        score = 0

        for tag in product["tags"]:
            # Exact tag match
            if tag in question_lower:
                score += 2

            # Individual word match
            for word in tag.split():
                if word in question_lower:
                    score += 1

        if score > 0:
            scored_products.append((score, product))

    # Highest score first
    scored_products.sort(key=lambda item: item[0], reverse=True)

    # Return only top_k products
    return [
        product
        for _, product in scored_products[:top_k]
    ]


# ============================================================
# 4. ASK MODEL USING CONTEXT
# ============================================================

def ask_model(question, context):
    """
    Sends the question and retrieved/stuffed context to the model.
    """

    prompt = f"""
Answer ONLY using the context provided below.

If the answer is not available in the context, say:
"I don't know based on the available information."

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=150,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.choices[0].message.content


# ============================================================
# 5. TOOL CALLING - ORDER STATUS
# ============================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": (
                "Look up the live shipping status of an order. "
                "Use this for any order-tracking question. "
                "Never guess an order status."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The customer's order ID, e.g. 8821",
                    }
                },
                "required": ["order_id"],
            },
        },
    }
]


# ============================================================
# 6. MOCK ORDER DATABASE
# ============================================================

def mock_get_order_status(order_id):
    """
    Simulates an external order database/API.
    """

    fake_orders = {
        "8821": "Out for delivery, expected today by 8pm"
    }

    return fake_orders.get(
        order_id,
        "No order found with that ID"
    )


# ============================================================
# 7. MODEL + TOOL CALLING
# ============================================================

def ask_model_with_tools(question):

    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    # First model call
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=200,
        tools=tools,
        messages=messages,
    )

    message = response.choices[0].message

    # If the model doesn't need a tool
    if not message.tool_calls:
        return message.content

    # Add the assistant's tool request to conversation
    messages.append(message)

    # Execute each requested tool
    for tool_call in message.tool_calls:

        arguments = json.loads(
            tool_call.function.arguments
        )

        order_id = arguments["order_id"]

        # Call our Python function
        status = mock_get_order_status(order_id)

        # Send tool result back to model
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(
                    {
                        "order_id": order_id,
                        "status": status,
                    }
                ),
            }
        )

    # Second model call
    final_response = client.chat.completions.create(
        model=MODEL,
        max_tokens=150,
        messages=messages,
    )

    return final_response.choices[0].message.content


# ============================================================
# 8. ROUTER
# ============================================================

def answer(question):

    question_lower = question.lower()

    # ----------------------------------------
    # TOOL CALLING
    # ----------------------------------------

    if any(
        word in question_lower
        for word in [
            "where is",
            "order #",
            "order status",
            "track",
        ]
    ):
        return ask_model_with_tools(question)

    # ----------------------------------------
    # RAG
    # ----------------------------------------

    if any(
        word in question_lower
        for word in [
            "suit",
            "recommend",
            "which product",
        ]
    ):

        relevant_products = retrieve_relevant_products(question)

        if not relevant_products:
            return "I don't know based on the available product information."

        context = "\n".join(
            f"- {product['name']} "
            f"({', '.join(product['tags'])})"
            for product in relevant_products
        )

        return ask_model(question, context)

    # ----------------------------------------
    # STUFFING
    # ----------------------------------------

    return ask_model(question, return_policy)


# ============================================================
# 9. TEST QUESTIONS
# ============================================================

questions = [
    "What's your return policy?",
    "Where is my order #8821?",
    "Which product suits a small kitchen?",
]


for question in questions:

    print("=" * 60)
    print(f"Q: {question}")
    print(f"A: {answer(question)}")